In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jrobischon/wikipedia-movie-plots")

print("Path to dataset files:", path)

100%|██████████| 29.9M/29.9M [00:03<00:00, 9.26MB/s]

Extracting files...


Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\jrobischon\wikipedia-movie-plots\versions\1


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print(model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [ ]:
import os
import pandas as pd

print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "wiki_movie_plots_deduped.csv"))
df.head()

['wiki_movie_plots_deduped.csv']


,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


In [6]:
df_title_plot = df[["Title", "Plot"]]
df_title_plot.head()

,Title,Plot
0,Kansas Saloon Smashers,"A bartender is working at a saloon, serving dr..."
1,Love by the Light of the Moon,"The moon, painted with a smiling face hangs ov..."
2,The Martyred Presidents,"The film, just over a minute long, is composed..."
3,"Terrible Teddy, the Grizzly King",Lasting just 61 seconds and consisting of two ...
4,Jack and the Beanstalk,The earliest known adaptation of the classic f...


In [7]:
df_title_plot.describe()

,Title,Plot
count,34886,34886
unique,32432,33869
top,Cinderella,"(マッスル人参争奪！超人大戦争, Massuru Ninjin Soudatsu! Chou..."
freq,8,6


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

df_title_plot = df[["Title", "Plot"]].copy()

embeddings = model.encode(
    df_title_plot["Plot"].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
df_title_plot["embedded_Plot"] = list(embeddings)


Batches:   0%|          | 0/546 [00:00<?, ?it/s]

In [13]:
import numpy as np

embedding_matrix = np.stack(df_title_plot["embedded_Plot"].to_numpy())

def similarity(Movie_Title:str, top_k:int=10):
    target_embedding = model.encode(Movie_Title)
    similarity_score = embedding_matrix @ target_embedding
    top_idx = np.argsort(-similarity_score)[:top_k]
    result = df_title_plot.iloc[top_idx][["Title"]].copy()
    result["similarity_score"] = similarity_score[top_idx]
    return result

In [14]:
similarity("Cinderella")

,Title,similarity_score
28772,Cinderella,0.657068
5078,Cinderella,0.626992
70,Cinderella,0.619944
22425,Charming,0.556959
9340,Cinderella,0.554377
18062,A Lowland Cinderella,0.551084
4128,Swing Shift Cinderella,0.541981
15224,Happily N'Ever After,0.494035
79,"His Majesty, the Scarecrow of Oz",0.462029
12152,Cinderella,0.455953
